<a href="https://colab.research.google.com/github/SaharMian/recsys-letterboxd/blob/main/Dataandeval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir('/content/drive/MyDrive/recsys'))

['films.csv', 'ratings.csv', 'sample_3000.parquet']


In [3]:
import pandas as pd

BASE = '/content/drive/MyDrive/recsys/'

ratings = pd.read_csv(BASE + 'ratings.csv', nrows=1000)
print(ratings.shape)
print(ratings.columns.tolist())
ratings.head()

(1000, 3)
['user_name', 'film_id', 'rating']


,user_name,film_id,rating
0,kurstboy,spider-man-2,4.5
1,kurstboy,tetsuo-the-iron-man,4.0
2,kurstboy,bram-stokers-dracula,4.5
3,kurstboy,poison-2023,4.5
4,kurstboy,the-rat-catcher-2023,3.5


In [4]:
BASE = '/content/drive/MyDrive/recsys/'

films = pd.read_csv(BASE + 'films.csv', engine='python', on_bad_lines='skip')
print(films.shape)
print(films.columns.tolist())
films.head()

(354834, 4)
['film_id', 'film_name', 'year', 'poster_url']


,film_id,film_name,year,poster_url
0,anthem-of-a-teenage-prophet,Anthem of a Teenage Prophet,2018.0,https://a.ltrbxd.com/resized/film-poster/3/8/6...
1,elf-buddys-musical-christmas,Elf: Buddy's Musical Christmas,2014.0,https://a.ltrbxd.com/resized/film-poster/2/3/8...
2,look-at-us-now-mother,"Look at Us Now, Mother!",2016.0,https://a.ltrbxd.com/resized/film-poster/3/2/6...
3,fate-grand-order-absolute-demonic-front-babylonia,Fate/Grand Order Absolute Demonic Front: Babyl...,2019.0,https://a.ltrbxd.com/resized/film-poster/6/7/1...
4,memory-2022,Memory,2022.0,https://a.ltrbxd.com/resized/film-poster/7/3/4...


***skillnad efter traisga rader***

In [5]:
with open(BASE + 'films.csv', encoding='utf-8') as f:
    n_lines = sum(1 for _ in f) - 1

print("Rader i filen:   ", n_lines)
print("Rader i films:   ", len(films))
print("Skillnad:        ", n_lines - len(films))

Rader i filen:    355141
Rader i films:    354834
Skillnad:         307


***user_name och film_id, textsträngar som upprepas, category= sparar unika namn en gång för att spara minne,float32 halverar minne istället för float64***

In [6]:
import os
print(f"{os.path.getsize(BASE + 'ratings.csv')/1e6:.0f} MB")

ratings = pd.read_csv(
    BASE + 'ratings.csv',
    dtype={'user_name': 'category', 'film_id': 'category', 'rating': 'float32'}
)
print(ratings.shape)
print(f"Minne: {ratings.memory_usage(deep=True).sum() /1e6:.0f} MB")

595 MB
(18175545, 3)
Minne: 215 MB


***betygsfördelning***

In [7]:
n_ratings = len(ratings)
n_users = ratings['user_name'].nunique()
n_films = ratings['film_id'].nunique()

print(f"Betyg:      {n_ratings:,}")
print(f"Användare:  {n_users:,}")
print(f"Filmer:     {n_films:,}")
print(f"Glesthet:   {1 - n_ratings / (n_users * n_films):.2%} av alla användare-film-par saknar betyg")

print(ratings['rating'].value_counts().sort_index())

Betyg:      18,175,545
Användare:  11,061
Filmer:     355,140
Glesthet:   99.54% av alla användare-film-par saknar betyg
rating
0.5     309105
1.0     615883
1.5     636460
2.0    1523578
2.5    1753268
3.0    3492302
3.5    3310119
4.0    3631351
4.5    1519156
5.0    1384323
Name: count, dtype: int64


***betyg per användare & per film***

In [8]:
per_user = ratings.groupby('user_name', observed=True).size()
per_film = ratings.groupby('film_id', observed=True).size()

print("Betyg per användare:")
print(per_user.describe())

print("\nBetyg per film:")
print(per_film.describe())

print(f"\nFilmer med bara 1 betyg: {(per_film == 1).mean():.1%}")
print(f"De 1000 mest betygsatta filmerna står för {per_film.nlargest(1000).sum() / len(ratings):.1%} av alla betyg")

Betyg per användare:
count    11061.000000
mean      1643.209927
std       1987.538284
min          1.000000
25%        562.000000
50%       1129.000000
75%       2088.000000
max      90755.000000
dtype: float64

Betyg per film:
count    355140.000000
mean         51.178513
std         307.481606
min           1.000000
25%           1.000000
50%           3.000000
75%          10.000000
max        8647.000000
dtype: float64

Filmer med bara 1 betyg: 31.6%
De 1000 mest betygsatta filmerna står för 24.8% av alla betyg


In [9]:
print("FILMER: minst N betyg")
for n in [3, 5, 10, 20, 50, 75]:
    kept = per_film[per_film >= n]
    print(f"  N={n:>3}: {len(kept):>7,} filmer ({len(kept)/len(per_film):.0%}), "
          f"{kept.sum()/len(ratings):.1%} av alla betyg kvar")

print("\nANVÄNDARE: minst N betyg")
for n in [20, 50, 100, 300, 562, 700, 1000]:
    kept = per_user[per_user >= n]
    print(f"  N={n:>3}: {len(kept):>6,} användare ({len(kept)/len(per_user):.0%}), "
          f"{kept.sum()/len(ratings):.1%} av alla betyg kvar")

print("\nDe 10 mest aktiva användarna:")
print(per_user.nlargest(10))
print(f"De står för {per_user.nlargest(10).sum()/len(ratings):.1%} av alla betyg")

FILMER: minst N betyg
  N=  3: 191,716 filmer (54%), 98.8% av alla betyg kvar
  N=  5: 142,374 filmer (40%), 97.9% av alla betyg kvar
  N= 10:  94,078 filmer (26%), 96.1% av alla betyg kvar
  N= 20:  61,091 filmer (17%), 93.7% av alla betyg kvar
  N= 50:  34,185 filmer (10%), 89.1% av alla betyg kvar
  N= 75:  26,503 filmer (7%), 86.6% av alla betyg kvar

ANVÄNDARE: minst N betyg
  N= 20: 10,983 användare (99%), 100.0% av alla betyg kvar
  N= 50: 10,882 användare (98%), 100.0% av alla betyg kvar
  N=100: 10,719 användare (97%), 99.9% av alla betyg kvar
  N=300:  9,750 användare (88%), 98.8% av alla betyg kvar
  N=562:  8,299 användare (75%), 95.4% av alla betyg kvar
  N=700:  7,545 användare (68%), 92.7% av alla betyg kvar
  N=1000:  6,099 användare (55%), 86.0% av alla betyg kvar

De 10 mest aktiva användarna:
user_name
acervofilmico      90755
punq               35541
ghettominibar      27236
loureviews         25021
thecinema4pylon    21652
allisoncm          21246
sbsiceland       

***sample***

In [10]:
import numpy as np

MIN_USER_RATINGS = 50
MAX_USER_RATINGS = 5000
N_USERS = 3000
MIN_FILM_RATINGS = 5
SEED = 42

rng = np.random.default_rng(SEED)

eligible = per_user[per_user >= MIN_USER_RATINGS].index.to_numpy()
chosen = rng.choice(eligible, size=N_USERS, replace=False)
sample = ratings[ratings['user_name'].isin(chosen)]

sample = sample.sample(frac=1, random_state=SEED)
sample = sample[sample.groupby('user_name', observed=True).cumcount() < MAX_USER_RATINGS]

film_counts = sample['film_id'].value_counts()
keep_films = film_counts[film_counts >= MIN_FILM_RATINGS].index
sample = sample[sample['film_id'].isin(keep_films)].copy()

for col in ['user_name', 'film_id']:
    sample[col] = sample[col].cat.remove_unused_categories()

print(f"Användare: {sample['user_name'].nunique():,}")
print(f"Filmer:    {sample['film_id'].nunique():,}")
print(f"Betyg:     {len(sample):,}  ({len(sample)/len(ratings):.1%} av originalet)")
print(f"Betyg per användare (median): {sample.groupby('user_name', observed=True).size().median():.0f}")

Användare: 3,000
Filmer:    61,555
Betyg:     4,439,328  (24.4% av originalet)
Betyg per användare (median): 1114


***Kontrollera att samplet är representativt, och spara det***

In [11]:
orig = ratings['rating'].value_counts(normalize=True).sort_index()
samp = sample['rating'].value_counts(normalize=True).sort_index()
print(pd.DataFrame({'original': orig, 'sampel': samp}).round(3))

sample.to_parquet(BASE + 'sample_3000.parquet', index=False)
print("Sparat!")

        original  sampel
rating                  
0.5        0.017   0.016
1.0        0.034   0.032
1.5        0.035   0.034
2.0        0.084   0.080
2.5        0.096   0.092
3.0        0.192   0.187
3.5        0.182   0.183
4.0        0.200   0.205
4.5        0.084   0.089
5.0        0.076   0.082
Sparat!


***Dela upp datan***

In [12]:
data = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)

grp = data.groupby('user_name', observed=True)
frac = grp.cumcount() / grp['rating'].transform('size')

data['split'] = 'train'
data.loc[frac >= 0.8, 'split'] = 'val'
data.loc[frac >= 0.9, 'split'] = 'test'

print(data['split'].value_counts())

test_liked = data[(data['split'] == 'test') & (data['rating'] >= 4.0)]
print(f"\nAnvändare med minst 1 gillad film i test: {test_liked['user_name'].nunique():,} av {data['user_name'].nunique():,}")

train_films = set(data.loc[data['split'] == 'train', 'film_id'].unique())
test_films = data.loc[data['split'] == 'test', 'film_id']
print(f"Test-betyg på filmer som saknas i train: {(~test_films.isin(train_films)).mean():.2%}")

data.to_parquet(BASE + 'split_3000.parquet', index=False)

split
train    3552662
val       444086
test      442580
Name: count, dtype: int64

Användare med minst 1 gillad film i test: 3,000 av 3,000
Test-betyg på filmer som saknas i train: 0.00%


***rekommenderare, och testar den på popularitetsbaselinen***

In [15]:
K = 10
LIKE = 4.0

train = data[data['split'] == 'train']
val   = data[data['split'] == 'val']
test  = data[data['split'] == 'test']

seen_train = train.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

def relevant_sets(df):
    liked = df[df['rating'] >= LIKE]
    return liked.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

def evaluate(recommend_fn, eval_df, seen):
    relevant = relevant_sets(eval_df)
    ndcgs, recalls, recommended = [], [], set()
    for user, rel in relevant.items():
        recs = recommend_fn(user, seen.get(user, set()), K)
        recommended.update(recs)
        hits = [1 if f in rel else 0 for f in recs]
        dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), K)))
        ndcgs.append(dcg / idcg)
        recalls.append(sum(hits) / len(rel))
    return {
        'NDCG@10': np.mean(ndcgs),
        'Recall@10': np.mean(recalls),
        'Täckning': len(recommended) / data['film_id'].nunique(),
    }


pop_rank = train[train['rating'] >= LIKE]['film_id'].value_counts().index.tolist()

def recommend_pop(user, seen, k):
    out = []
    for f in pop_rank:
        if f not in seen:
            out.append(f)
            if len(out) == k:
                break
    return out

print(evaluate(recommend_pop, val, seen_train))

{'NDCG@10': np.float64(0.1841454716946846), 'Recall@10': np.float64(0.03860758890220193), 'Täckning': 0.001364633254812769}
